In [ ]:
!pip -q install pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg ultralytics albumentations


In [ ]:
import os
import cv2
import yaml
import time
import random
import numpy as np
import pandas as pd
import pydicom
import albumentations as A

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


# Config

In [ ]:
# =========================
# PATH CONFIG
# =========================
from pathlib import Path

ROOT = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection")

TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"
TRAIN_CSV = Path('/kaggle/input/datasets/benxelua/clean-dataset/annotations_train_merged_other_500nf.csv')
TEST_CSV = Path('/kaggle/input/datasets/benxelua/clean-dataset/annotations_test_merged_other_500nf.csv')

# output YOLO dataset
OUT_ROOT = Path("/kaggle/working/yolo_vindr_multiclass")

PNG_DIR = OUT_ROOT / "images"
LBL_DIR = OUT_ROOT / "labels"

IMG_TRAIN_DIR = PNG_DIR / "train"
IMG_VAL_DIR   = PNG_DIR / "val"
IMG_TEST_DIR  = PNG_DIR / "test"
LBL_TRAIN_DIR = LBL_DIR / "train"
LBL_VAL_DIR   = LBL_DIR / "val"
LBL_TEST_DIR  = LBL_DIR / "test"

for d in [IMG_TRAIN_DIR, IMG_VAL_DIR, IMG_TEST_DIR, LBL_TRAIN_DIR, LBL_VAL_DIR, LBL_TEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# SUBSET CONFIG
# =========================
USE_SMALL_SUBSET = False     # True = subset nhỏ, False = full dataset
IMAGES_PER_CLASS = 2         # mỗi class lấy khoảng N ảnh trong small subset
NO_FINDING_IMAGES = 1000      # giữ khoảng 500-1000 ảnh No finding/background; đổi 500/1000 nếu muốn
SEED = 42

# Sau khi gộp train + test gốc, chia lại theo image-level multilabel stratification.
SPLIT_RATIOS = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15,
}

# =========================
# IMAGE CONFIG
# =========================
IMG_SIZE = 512              # có thể đổi 640 nếu muốn
SAVE_AS_JPG = False         # True nhanh hơn PNG
JPG_QUALITY = 95

# Helper functions

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)


def find_dicom_files_for_image_ids(folder: Path, image_ids):
    requested_ids = sorted({str(x).strip() for x in image_ids if pd.notna(x) and str(x).strip()})
    matched = {}

    for image_id in requested_ids:
        direct_candidates = [
            folder / f"{image_id}.dicom",
            folder / f"{image_id}.dcm",
            folder / image_id,
        ]
        for candidate in direct_candidates:
            if candidate.is_file():
                matched[image_id] = candidate
                break

    remaining = [image_id for image_id in requested_ids if image_id not in matched]
    if remaining:
        remaining_set = set(remaining)
        for p in sorted(folder.rglob("*")):
            if not remaining_set:
                break
            if p.is_file() and p.suffix.lower() in {".dicom", ".dcm"} and p.stem in remaining_set:
                matched[p.stem] = p
                remaining_set.remove(p.stem)

    missing = [image_id for image_id in requested_ids if image_id not in matched]
    matched_files = [matched[image_id] for image_id in requested_ids if image_id in matched]
    return matched_files, matched, missing


from pathlib import Path
import numpy as np
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut

def read_dicom_to_uint8(path: Path):
    ds = pydicom.dcmread(str(path))

    # 1. Rescale / Modality LUT
    img = apply_modality_lut(ds.pixel_array, ds).astype(np.float32)

    # 2. Handle MONOCHROME1
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    # 3. Min-max normalization
    img_min = img.min()
    img_max = img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)
    else:
        img = np.zeros_like(img)

    # 4. Convert to uint8
    return (img * 255.0).clip(0, 255).astype(np.uint8)

from pathlib import Path
import numpy as np
import pydicom

# def read_dicom_to_uint8(path: Path):
#     ds = pydicom.dcmread(str(path))
#     
#     # 1. Read raw pixel array only
#     img = ds.pixel_array.astype(np.float32)

#     # 2. Handle Photometric Interpretation
#     if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
#         img = np.max(img) - img

#     # 3. Raw min-max normalization
#     img = img - np.min(img)
#     max_val = np.max(img)
#     if max_val > 0:
#         img = img / max_val

#     # 4. Convert to uint8
#     img = (img * 255.0).clip(0, 255).astype(np.uint8)
#     return img

def resize_image_keep_shape(img, size=1024):
    h, w = img.shape[:2]
    resized = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return resized, w, h


# Đọc CSV và chuẩn hóa annotation

In [ ]:
# Label fusion helpers for VinDr-CXR training annotations
from dataclasses import dataclass

@dataclass(frozen=True)
class FusionConfig:
    iou_threshold: float = 0.5
    ios_threshold: float = 0.7
    merge_mode: str = "median"  # options: median, mean, enclosing


def _box_area(box):
    width = max(0.0, float(box[2]) - float(box[0]))
    height = max(0.0, float(box[3]) - float(box[1]))
    return width * height


def _intersection_area(box_a, box_b):
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    return width * height


def compute_iou(box_a, box_b):
    inter = _intersection_area(box_a, box_b)
    if inter <= 0.0:
        return 0.0
    union = _box_area(box_a) + _box_area(box_b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def compute_ios(box_a, box_b):
    inter = _intersection_area(box_a, box_b)
    if inter <= 0.0:
        return 0.0
    smaller = min(_box_area(box_a), _box_area(box_b))
    if smaller <= 0.0:
        return 0.0
    return inter / smaller


def should_link_boxes(box_a, box_b, config):
    return (
        compute_iou(box_a, box_b) >= config.iou_threshold
        or compute_ios(box_a, box_b) >= config.ios_threshold
    )


def merge_cluster_boxes(boxes, merge_mode="median"):
    boxes = np.asarray(boxes, dtype=float)
    if len(boxes) == 1:
        return boxes[0].astype(float)

    if merge_mode == "median":
        merged = np.median(boxes, axis=0)
    elif merge_mode == "mean":
        merged = np.mean(boxes, axis=0)
    elif merge_mode == "enclosing":
        merged = np.array([
            np.min(boxes[:, 0]),
            np.min(boxes[:, 1]),
            np.max(boxes[:, 2]),
            np.max(boxes[:, 3]),
        ], dtype=float)
    else:
        raise ValueError(f"Unsupported merge_mode: {merge_mode}")

    if merged[2] <= merged[0]:
        merged[2] = merged[0] + 1e-6
    if merged[3] <= merged[1]:
        merged[3] = merged[1] + 1e-6
    return merged.astype(float)


def _connected_components(indices, adjacency):
    components = []
    visited = set()

    for start in indices:
        if start in visited:
            continue
        stack = [start]
        component = []
        visited.add(start)

        while stack:
            node = stack.pop()
            component.append(node)
            for neighbor in adjacency[node]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

        components.append(sorted(component))

    return components


def fuse_same_class_boxes(class_df, bbox_cols=("x_min", "y_min", "x_max", "y_max"), config=None):
    config = config or FusionConfig()
    if len(class_df) <= 1:
        out = class_df.copy()
        out["fusion_group_size"] = 1
        out["was_fused"] = False
        return out

    coords = class_df.loc[:, list(bbox_cols)].to_numpy(dtype=float)
    indices = list(range(len(class_df)))
    adjacency = {idx: set() for idx in indices}

    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            if should_link_boxes(coords[i], coords[j], config):
                adjacency[i].add(j)
                adjacency[j].add(i)

    components = _connected_components(indices, adjacency)
    fused_rows = []

    for component in components:
        component_df = class_df.iloc[component].copy()
        merged_box = merge_cluster_boxes(
            component_df.loc[:, list(bbox_cols)].to_numpy(dtype=float),
            merge_mode=config.merge_mode,
        )

        row = component_df.iloc[0].copy()
        row[bbox_cols[0]] = float(merged_box[0])
        row[bbox_cols[1]] = float(merged_box[1])
        row[bbox_cols[2]] = float(merged_box[2])
        row[bbox_cols[3]] = float(merged_box[3])
        row["fusion_group_size"] = int(len(component))
        row["was_fused"] = bool(len(component) > 1)
        fused_rows.append(row)

    return pd.DataFrame(fused_rows).reset_index(drop=True)


def fuse_annotation_dataframe(
    df,
    image_col="image_id",
    class_col="class_name",
    bbox_cols=("x_min", "y_min", "x_max", "y_max"),
    config=None,
):
    config = config or FusionConfig()
    fused_groups = []
    grouped = df.groupby([image_col, class_col], sort=False, group_keys=False)
    for _, group_df in grouped:
        fused_groups.append(
            fuse_same_class_boxes(
                class_df=group_df,
                bbox_cols=bbox_cols,
                config=config,
            )
        )

    if not fused_groups:
        out = df.copy()
        out["fusion_group_size"] = 1
        out["was_fused"] = False
        return out

    return pd.concat(fused_groups, ignore_index=True).reset_index(drop=True)



In [ ]:
def normalize_annotation_csv(csv_path: Path, source_name: str):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Không thấy annotation CSV cho {source_name}: {csv_path}")

    out = pd.read_csv(csv_path)
    print(f"{source_name} CSV shape:", out.shape)
    print(f"{source_name} CSV columns:", out.columns.tolist())
    display(out.head())

    required_cols = ["image_id", "class_name", "x_min", "y_min", "x_max", "y_max"]
    missing_cols = [c for c in required_cols if c not in out.columns]
    if missing_cols:
        raise ValueError(f"Thiếu cột trong {source_name} CSV: {missing_cols}")

    out = out.copy()
    out["source"] = source_name
    out["image_id"] = out["image_id"].astype("string").str.strip()
    out["class_name"] = out["class_name"].astype("string").str.strip()
    out = out.dropna(subset=["image_id", "class_name"]).copy()

    for c in ["x_min", "y_min", "x_max", "y_max"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # sample_id là khóa duy nhất sau khi trộn train/test, tránh trùng image_id giữa 2 nguồn.
    out["sample_id"] = out["source"].astype(str) + "__" + out["image_id"].astype(str)
    return out.reset_index(drop=True)


train_csv_all = normalize_annotation_csv(TRAIN_CSV, "train")
test_csv_all = normalize_annotation_csv(TEST_CSV, "test")
df_csv_all = pd.concat([train_csv_all, test_csv_all], ignore_index=True)

all_sample_ids = set(df_csv_all["sample_id"].dropna().astype(str).str.strip().unique().tolist())
all_csv_ids = all_sample_ids  # giữ tên cũ để các cell sau vẫn dễ đọc
no_finding_sample_ids = set(
    df_csv_all.loc[
        df_csv_all["class_name"].fillna("").str.lower() == "no finding",
        "sample_id",
    ].astype(str)
)

print("Merged CSV shape:", df_csv_all.shape)
print("Merged unique sample_id:", len(all_sample_ids))
print("No finding/background sample_id:", len(no_finding_sample_ids))
print("Merged source distribution:")
display(df_csv_all.drop_duplicates("sample_id")["source"].value_counts().rename_axis("source").reset_index(name="images"))

# Không export "No finding" thành class bbox cho YOLO detection.
# Ảnh No finding vẫn được giữ trong all_sample_ids, được split/export như background với file label rỗng.
df_full = df_csv_all.copy()
df_full = df_full[df_full["class_name"].fillna("").str.lower() != "no finding"].copy()
df_full = df_full.dropna(subset=["image_id", "sample_id", "class_name", "x_min", "y_min", "x_max", "y_max"]).copy()
df_full = df_full[(df_full["x_max"] > df_full["x_min"]) & (df_full["y_max"] > df_full["y_min"])].copy()

# =========================
# OPTIONAL LABEL FUSION
# =========================
# VinDr-CXR có thể có nhiều bbox chồng nhau cho cùng 1 lesion vì được annotate bởi nhiều radiologists.
# Bước này chỉ clean annotations trước khi export label.
# Nó KHÔNG phải post-processing ở inference time.
BOX_FUSION_ENABLED = True
BOX_FUSION_IOU_THRESHOLD = 0.5
BOX_FUSION_IOS_THRESHOLD = 0.7
BOX_FUSION_MERGE_MODE = "median"   # options: median, mean, enclosing

df_before_fusion = df_full.copy()

if BOX_FUSION_ENABLED:
    before_fusion_boxes = len(df_full)
    before_fusion_images = df_full["sample_id"].nunique()
    fusion_config = FusionConfig(
        iou_threshold=BOX_FUSION_IOU_THRESHOLD,
        ios_threshold=BOX_FUSION_IOS_THRESHOLD,
        merge_mode=BOX_FUSION_MERGE_MODE,
    )
    df_full = fuse_annotation_dataframe(
        df_full,
        image_col="sample_id",
        class_col="class_name",
        config=fusion_config,
    )
    print("Label fusion enabled")
    print("Fusion config:", fusion_config)
    print("Boxes before fusion:", before_fusion_boxes)
    print("Boxes after fusion:", len(df_full))
    print("Images with abnormal boxes before fusion:", before_fusion_images)
    print("Images with abnormal boxes after fusion:", df_full["sample_id"].nunique())
    print("Rows created from merged clusters:", int(df_full["was_fused"].sum()))
    display(
        df_full.loc[df_full["was_fused"], ["source", "image_id", "sample_id", "class_name", "fusion_group_size"]]
        .head(10)
        .reset_index(drop=True)
    )

# Build class list từ train + test đã gộp.
CLASS_NAMES = sorted(df_full["class_name"].unique().tolist())
CLASS2ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID2CLASS = {i: name for name, i in CLASS2ID.items()}

abnormal_sample_ids = set(df_full["sample_id"].astype(str).unique())
background_sample_ids = sorted(all_sample_ids - abnormal_sample_ids)
no_finding_only_sample_ids = sorted(set(no_finding_sample_ids) & set(background_sample_ids))

print("Num classes:", len(CLASS_NAMES))
print("CLASS_NAMES:", CLASS_NAMES)
print("CLASS2ID:", CLASS2ID)
print("Total unique sample_id in merged CSV:", len(all_sample_ids))
print("Images with >=1 valid abnormal bbox:", len(abnormal_sample_ids))
print("Background-only sample_id:", len(background_sample_ids))
print("No finding only sample_id:", len(no_finding_only_sample_ids))

In [ ]:
# # Debug visualize label fusion before vs after for one training image
# import matplotlib.pyplot as plt
# from matplotlib.patches import Rectangle

# DEBUG_FUSION_IMAGE_ID = ""  # ví dụ: "0005e8e3701dfb1dd93d53e2ff537b6e"
# SHOW_CLASS_TEXT = True
# FIGSIZE = (18, 9)

# if not DEBUG_FUSION_IMAGE_ID:
#     raise ValueError("Điền DEBUG_FUSION_IMAGE_ID trước khi chạy cell này")
# if 'df_before_fusion' not in globals():
#     raise ValueError("Chạy cell fusion trước để tạo df_before_fusion và df_full")


# def find_train_dicom_path(train_dir: Path, image_id: str):
#     direct_candidates = [
#         train_dir / f"{image_id}.dicom",
#         train_dir / f"{image_id}.dcm",
#         train_dir / image_id,
#     ]
#     for candidate in direct_candidates:
#         if candidate.is_file():
#             return candidate

#     for p in sorted(train_dir.rglob("*")):
#         if p.is_file() and p.suffix.lower() in {".dicom", ".dcm"} and p.stem == image_id:
#             return p
#     return None


# def draw_boxes(ax, boxes_df, color, title_prefix=""):
#     for _, row in boxes_df.iterrows():
#         x1, y1, x2, y2 = float(row['x_min']), float(row['y_min']), float(row['x_max']), float(row['y_max'])
#         rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
#         ax.add_patch(rect)
#         if SHOW_CLASS_TEXT:
#             label = str(row['class_name'])
#             if 'fusion_group_size' in row and pd.notna(row.get('fusion_group_size', np.nan)):
#                 if int(row.get('fusion_group_size', 1)) > 1:
#                     label = f"{label} (n={int(row['fusion_group_size'])})"
#             ax.text(
#                 x1,
#                 max(5.0, y1 - 4.0),
#                 f"{title_prefix}{label}",
#                 color=color,
#                 fontsize=9,
#                 bbox=dict(facecolor='black', alpha=0.5, pad=1),
#             )


# before_df = df_before_fusion[df_before_fusion['image_id'].astype(str) == str(DEBUG_FUSION_IMAGE_ID)].copy()
# after_df = df_full[df_full['image_id'].astype(str) == str(DEBUG_FUSION_IMAGE_ID)].copy()

# print(f"image_id = {DEBUG_FUSION_IMAGE_ID}")
# print(f"Boxes before fusion: {len(before_df)}")
# print(f"Boxes after fusion: {len(after_df)}")

# if len(before_df) == 0 and len(after_df) == 0:
#     raise ValueError(f"Không tìm thấy annotation cho image_id={DEBUG_FUSION_IMAGE_ID}")

# display(before_df[['image_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max']].reset_index(drop=True))
# display(after_df[['image_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max', 'fusion_group_size', 'was_fused']].reset_index(drop=True))

# dicom_path = find_train_dicom_path(TRAIN_DIR, str(DEBUG_FUSION_IMAGE_ID))
# if dicom_path is None:
#     raise FileNotFoundError(f"Không tìm thấy file DICOM cho image_id={DEBUG_FUSION_IMAGE_ID} trong {TRAIN_DIR}")

# img = read_dicom_to_uint8(dicom_path)

# fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
# for ax in axes:
#     if img.ndim == 2:
#         ax.imshow(img, cmap='gray')
#     else:
#         ax.imshow(img)
#     ax.axis('off')

# axes[0].set_title(f"Before fusion: {DEBUG_FUSION_IMAGE_ID}")
# draw_boxes(axes[0], before_df, color='red', title_prefix='before: ')

# axes[1].set_title(f"After fusion: {DEBUG_FUSION_IMAGE_ID}")
# draw_boxes(axes[1], after_df, color='lime', title_prefix='after: ')

# plt.tight_layout()
# plt.show()



# Chọn subset nhỏ theo từng class từ train + test đã gộp

In [ ]:
import pandas as pd
import numpy as np

print("Building subset from merged train + test CSV...")

# subset theo annotation bbox hợp lệ, nhưng full mode vẫn lấy toàn bộ sample_id từ CSV.
df_work = df_full.copy()

class_to_ids = (
    df_work.groupby("class_name")["sample_id"]
    .unique()
    .to_dict()
)


def build_small_subset_by_class_fast(class_to_ids, images_per_class=12, seed=42, background_ids=None):
    rng = np.random.default_rng(seed)

    selected_ids = set()
    per_class_summary = []

    for cls in sorted(class_to_ids.keys()):
        cls_ids = list(class_to_ids[cls])
        n_take = min(images_per_class, len(cls_ids))

        if n_take > 0:
            chosen = rng.choice(cls_ids, size=n_take, replace=False).tolist()
        else:
            chosen = []

        selected_ids.update(chosen)

        per_class_summary.append({
            "class_name": cls,
            "available_images": len(cls_ids),
            "selected_images": len(chosen)
        })

    background_ids = sorted(set(background_ids or []))
    n_take_bg = min(images_per_class, len(background_ids))
    if n_take_bg > 0:
        chosen_bg = rng.choice(background_ids, size=n_take_bg, replace=False).tolist()
    else:
        chosen_bg = []
    selected_ids.update(chosen_bg)
    per_class_summary.append({
        "class_name": "No finding",
        "available_images": len(background_ids),
        "selected_images": len(chosen_bg),
    })

    return selected_ids, pd.DataFrame(per_class_summary)


rng_subset = np.random.default_rng(SEED)
background_pool = sorted(set(background_sample_ids))
n_no_finding_take = min(int(NO_FINDING_IMAGES), len(background_pool))
selected_background_ids = set(
    rng_subset.choice(background_pool, size=n_no_finding_take, replace=False).tolist()
) if n_no_finding_take > 0 else set()

if USE_SMALL_SUBSET:
    chosen_ids, subset_summary_df = build_small_subset_by_class_fast(
        class_to_ids,
        images_per_class=IMAGES_PER_CLASS,
        seed=SEED,
        background_ids=background_pool,
    )
    chosen_ids = set(map(str, chosen_ids)) & all_sample_ids
    print("Using SMALL subset mode")
    print("Total selected unique samples from merged CSV:", len(chosen_ids))
    display(subset_summary_df)
else:
    # Full mode: giữ toàn bộ ảnh có bbox bất thường, nhưng chỉ lấy một lượng No finding/background vừa phải.
    chosen_ids = set(abnormal_sample_ids) | selected_background_ids
    print("Using FULL dataset mode with capped No finding/background")
    print("Abnormal samples selected:", len(abnormal_sample_ids))
    print("Background/No finding samples selected:", len(selected_background_ids), "of", len(background_pool))
    print("NO_FINDING_IMAGES:", NO_FINDING_IMAGES)
    print("Total selected unique samples from merged CSV:", len(chosen_ids))

print("Matching requested DICOM files from both source folders...")

SOURCE_DIRS = {
    "train": TRAIN_DIR,
    "test": TEST_DIR,
}

sample_lookup_df = (
    df_csv_all[["sample_id", "source", "image_id"]]
    .drop_duplicates("sample_id")
    .reset_index(drop=True)
)


def find_dicom_records_for_sample_ids(sample_ids, sample_lookup_df, source_dirs):
    requested = set(map(str, sample_ids))
    lookup = sample_lookup_df[sample_lookup_df["sample_id"].isin(requested)].copy()

    records = []
    path_map = {}
    missing_sample_ids = []

    for source, group in lookup.groupby("source", sort=False):
        source_dir = Path(source_dirs[source])
        if not source_dir.exists():
            missing_sample_ids.extend(group["sample_id"].astype(str).tolist())
            continue

        image_ids = group["image_id"].astype(str).tolist()
        _, image_path_map, missing_image_ids = find_dicom_files_for_image_ids(source_dir, image_ids)
        missing_image_ids = set(missing_image_ids)

        for row in group.itertuples(index=False):
            sample_id = str(row.sample_id)
            image_id = str(row.image_id)
            if image_id in image_path_map:
                record = {
                    "sample_id": sample_id,
                    "source": str(row.source),
                    "image_id": image_id,
                    "path": image_path_map[image_id],
                }
                records.append(record)
                path_map[sample_id] = image_path_map[image_id]
            elif image_id in missing_image_ids:
                missing_sample_ids.append(sample_id)

    records = sorted(records, key=lambda x: x["sample_id"])
    missing_sample_ids = sorted(set(missing_sample_ids) | (requested - set(path_map.keys())))
    return records, path_map, missing_sample_ids


dicom_records_selected, dicom_path_map, missing_dicom_ids = find_dicom_records_for_sample_ids(
    chosen_ids,
    sample_lookup_df=sample_lookup_df,
    source_dirs=SOURCE_DIRS,
)
dicom_files_selected = [record["path"] for record in dicom_records_selected]
used_sample_ids = sorted(dicom_path_map.keys())
used_image_ids = used_sample_ids  # alias cũ, giờ là sample_id

print("Requested merged sample_ids:", len(chosen_ids))
print("Matched DICOM records:", len(dicom_records_selected))
print("Missing requested DICOM samples:", len(missing_dicom_ids))
if missing_dicom_ids:
    print("First missing sample_ids:", missing_dicom_ids[:10])

source_summary = pd.DataFrame(dicom_records_selected)["source"].value_counts().rename_axis("source").reset_index(name="matched_images")
display(source_summary)

In [ ]:
used_sample_ids = sorted(dicom_path_map.keys())
used_image_ids = used_sample_ids  # alias cũ, giờ là sample_id

df = df_full[df_full["sample_id"].astype(str).str.strip().isin(used_sample_ids)].copy()
used_negative_only_ids = sorted(set(used_sample_ids) - set(df["sample_id"].astype(str).unique()))

print("Requested merged sample_ids:", len(chosen_ids))
print("Used DICOM records:", len(dicom_records_selected))
print("Selected annotation rows:", len(df))
print("Used annotated sample_id:", df["sample_id"].nunique())
print("Used negative-only sample_id:", len(used_negative_only_ids))
print("Missing requested sample_ids:", len(missing_dicom_ids))

# Stratified split train/val/test

In [ ]:
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

BACKGROUND_CLASS = "No finding"


def normalize_split_ratios(split_ratios):
    split_ratios = dict(split_ratios)
    total = sum(float(v) for v in split_ratios.values())
    if total <= 0:
        raise ValueError("SPLIT_RATIOS phải có tổng > 0")
    return {k: float(v) / total for k, v in split_ratios.items()}


def compute_target_sizes(n_items, split_ratios):
    raw = {split: n_items * ratio for split, ratio in split_ratios.items()}
    sizes = {split: int(np.floor(value)) for split, value in raw.items()}
    remaining = n_items - sum(sizes.values())

    order = sorted(split_ratios.keys(), key=lambda split: (raw[split] - sizes[split], split), reverse=True)
    for split in order[:remaining]:
        sizes[split] += 1

    nonzero_splits = [split for split, ratio in split_ratios.items() if ratio > 0]
    if n_items >= len(nonzero_splits):
        for split in nonzero_splits:
            if sizes[split] == 0:
                donor = max(sizes, key=sizes.get)
                sizes[donor] -= 1
                sizes[split] += 1

    return sizes


def build_image_class_sets(df, image_ids, image_col="sample_id", class_col="class_name"):
    image_ids = sorted(set(map(str, image_ids)))
    grouped = (
        df.groupby(image_col)[class_col]
        .apply(lambda x: sorted(set(map(str, x))))
        .to_dict()
    )
    return {
        image_id: grouped.get(image_id, [BACKGROUND_CLASS])
        for image_id in image_ids
    }


def build_multilabel_stratified_split(df, image_ids, split_ratios=None, seed=42):
    split_ratios = normalize_split_ratios(split_ratios or {"train": 0.7, "val": 0.15, "test": 0.15})
    rng = np.random.default_rng(seed)

    image_ids = sorted(set(map(str, image_ids)))
    image_class_sets = build_image_class_sets(df, image_ids)
    split_names = list(split_ratios.keys())

    target_sizes = compute_target_sizes(len(image_ids), split_ratios)
    class_total = Counter(cls for classes in image_class_sets.values() for cls in classes)
    target_class_counts = {
        split: {cls: class_total[cls] * split_ratios[split] for cls in class_total}
        for split in split_names
    }

    current_sizes = {split: 0 for split in split_names}
    current_class_counts = {split: Counter() for split in split_names}
    split_sets = {split: set() for split in split_names}

    # Ưu tiên gán ảnh chứa class hiếm/multilabel trước để giữ phân bố class tốt hơn.
    ordered_ids = sorted(
        image_ids,
        key=lambda image_id: (
            -sum(1.0 / max(class_total[cls], 1) for cls in image_class_sets[image_id]),
            -len(image_class_sets[image_id]),
            rng.random(),
        ),
    )

    for image_id in ordered_ids:
        classes = image_class_sets[image_id]
        candidates = [split for split in split_names if current_sizes[split] < target_sizes[split]]
        if not candidates:
            candidates = split_names

        best_split = None
        best_score = None

        for split in candidates:
            size_deficit = (target_sizes[split] - current_sizes[split]) / max(target_sizes[split], 1)
            class_deficit = 0.0
            over_penalty = 0.0

            for cls in classes:
                target = max(target_class_counts[split][cls], 1e-9)
                before = current_class_counts[split][cls]
                after = before + 1
                class_deficit += max(target - before, 0.0) / target
                over_penalty += max(after - target, 0.0) / target

            score = size_deficit + class_deficit - over_penalty
            tie_breaker = rng.random() * 1e-6
            score += tie_breaker

            if best_score is None or score > best_score:
                best_score = score
                best_split = split

        split_sets[best_split].add(image_id)
        current_sizes[best_split] += 1
        for cls in classes:
            current_class_counts[best_split][cls] += 1

    return split_sets, image_class_sets


def summarize_split_distribution(split_sets, image_class_sets):
    class_names = sorted({cls for classes in image_class_sets.values() for cls in classes})
    rows = []
    for cls in class_names:
        total = sum(1 for classes in image_class_sets.values() if cls in classes)
        row = {"class_name": cls, "total_images": total}
        for split, ids in split_sets.items():
            count = sum(1 for image_id in ids if cls in image_class_sets[image_id])
            row[f"{split}_images"] = count
            row[f"{split}_pct_of_class"] = round(100.0 * count / total, 2) if total else 0.0
        rows.append(row)
    return pd.DataFrame(rows).sort_values("class_name").reset_index(drop=True)

In [ ]:
image_ids_selected = used_sample_ids.copy()

split_sets, image_class_sets = build_multilabel_stratified_split(
    df=df,
    image_ids=image_ids_selected,
    split_ratios=SPLIT_RATIOS,
    seed=SEED,
)

train_ids = split_sets["train"]
val_ids = split_sets["val"]
test_ids = split_sets["test"]
split_distribution_df = summarize_split_distribution(split_sets, image_class_sets)

print("Used images for split:", len(image_ids_selected))
print("Split ratios:", normalize_split_ratios(SPLIT_RATIOS))
print("Train images:", len(train_ids))
print("Val images:", len(val_ids))
print("Test images:", len(test_ids))
print("Overlap train/val:", len(train_ids & val_ids))
print("Overlap train/test:", len(train_ids & test_ids))
print("Overlap val/test:", len(val_ids & test_ids))

display(split_distribution_df)

# Lưu metadata để lần sau dùng lại

In [ ]:
import json

meta = {
    "use_small_subset": USE_SMALL_SUBSET,
    "images_per_class": IMAGES_PER_CLASS,
    "no_finding_images": NO_FINDING_IMAGES,
    "selected_background_ids": sorted(list(selected_background_ids)),
    "seed": SEED,
    "split_ratios": normalize_split_ratios(SPLIT_RATIOS),
    "img_size": IMG_SIZE,
    "save_as_jpg": SAVE_AS_JPG,
    "jpg_quality": JPG_QUALITY,
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "chosen_sample_ids": sorted(list(chosen_ids)),
    "used_sample_ids": sorted(list(used_sample_ids)),
    "missing_dicom_sample_ids": sorted(list(missing_dicom_ids)),
    "train_ids": sorted(list(train_ids)),
    "val_ids": sorted(list(val_ids)),
    "test_ids": sorted(list(test_ids)),
}

with open(OUT_ROOT / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

split_distribution_df.to_csv(OUT_ROOT / "split_distribution.csv", index=False)

print("Saved:", OUT_ROOT / "meta.json")
print("Saved:", OUT_ROOT / "split_distribution.csv")

# Build annotation map nhanh

In [ ]:
from collections import defaultdict

ann_map = defaultdict(list)

for r in df.itertuples(index=False):
    class_id = CLASS2ID[r.class_name]
    ann_map[str(r.sample_id)].append(
        (r.x_min, r.y_min, r.x_max, r.y_max, class_id)
    )

split_map = {}
for x in train_ids:
    split_map[str(x)] = "train"
for x in val_ids:
    split_map[str(x)] = "val"
for x in test_ids:
    split_map[str(x)] = "test"

print("Images with annotations:", len(ann_map))
print("Images without annotations:", len(set(split_map) - set(ann_map)))
print("Images in split map:", len(split_map))
print("Split map counts:", pd.Series(split_map).value_counts().to_dict())

# Export ảnh + label nhanh, có track time

In [ ]:
import albumentations as A

DICOM_TRAIN_AUG = A.Compose(
    [
        # Random Horizontal Flip
        A.HorizontalFlip(p=0.5),

        # Sharpness Enhancement (gần với mô tả dùng Gaussian để làm rõ ảnh)
        A.UnsharpMask(
            blur_limit=(3, 5),
            sigma_limit=(0.1, 1.5),
            alpha=(0.1, 0.3),
            threshold=10,
            p=0.20
        ),

        # Gaussian Blur
        A.GaussianBlur(
            blur_limit=(3, 5),
            sigma_limit=(0.1, 1.5),
            p=0.20
        ),

        # Image Histogram Equalization
        A.Equalize(
            mode="cv",
            by_channels=False,
            p=0.20
        ),

        # Random Brightness only
        A.RandomBrightnessContrast(
            brightness_limit=0.10,
            contrast_limit=0.0,
            p=0.20
        ),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["class_labels"],
        min_visibility=0.0,
    ),
)


def sanitize_pascal_voc_boxes(boxes, width, height):
    sanitized = []
    for (x1, y1, x2, y2, class_id) in boxes:
        x1 = float(np.clip(x1, 0, width))
        y1 = float(np.clip(y1, 0, height))
        x2 = float(np.clip(x2, 0, width))
        y2 = float(np.clip(y2, 0, height))

        if x2 <= x1 or y2 <= y1:
            continue

        sanitized.append((x1, y1, x2, y2, class_id))

    return sanitized


def apply_dicom_augmentation(img, boxes, split):
    height, width = img.shape[:2]
    boxes = sanitize_pascal_voc_boxes(boxes, width=width, height=height)

    if split != "train" or len(boxes) == 0:
        return img, boxes

    bboxes = [(x1, y1, x2, y2) for (x1, y1, x2, y2, _) in boxes]
    class_labels = [class_id for (_, _, _, _, class_id) in boxes]

    transformed = DICOM_TRAIN_AUG(
        image=img,
        bboxes=bboxes,
        class_labels=class_labels,
    )

    aug_boxes = [
        (x1, y1, x2, y2, class_id)
        for (x1, y1, x2, y2), class_id in zip(
            transformed["bboxes"],
            transformed["class_labels"],
        )
    ]
    aug_boxes = sanitize_pascal_voc_boxes(aug_boxes, width=width, height=height)
    return transformed["image"], aug_boxes


def build_yolo_lines(boxes, orig_w, orig_h, img_size):
    yolo_lines = []

    if len(boxes) == 0:
        return yolo_lines

    sx = img_size / orig_w
    sy = img_size / orig_h

    for (x1, y1, x2, y2, class_id) in boxes:
        x1 *= sx
        x2 *= sx
        y1 *= sy
        y2 *= sy

        xc = ((x1 + x2) / 2.0) / img_size
        yc = ((y1 + y2) / 2.0) / img_size
        bw = (x2 - x1) / img_size
        bh = (y2 - y1) / img_size

        xc = min(max(xc, 0.0), 1.0)
        yc = min(max(yc, 0.0), 1.0)
        bw = min(max(bw, 0.0), 1.0)
        bh = min(max(bh, 0.0), 1.0)

        if bw > 0 and bh > 0:
            yolo_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    return yolo_lines


def save_image_and_label(img, boxes, image_id, split):
    img_resized, orig_w, orig_h = resize_image_keep_shape(img, size=IMG_SIZE)

    ext = ".jpg" if SAVE_AS_JPG else ".png"

    if split == "train":
        img_out_path = IMG_TRAIN_DIR / f"{image_id}{ext}"
        lbl_out_path = LBL_TRAIN_DIR / f"{image_id}.txt"
    elif split == "val":
        img_out_path = IMG_VAL_DIR / f"{image_id}{ext}"
        lbl_out_path = LBL_VAL_DIR / f"{image_id}.txt"
    elif split == "test":
        img_out_path = IMG_TEST_DIR / f"{image_id}{ext}"
        lbl_out_path = LBL_TEST_DIR / f"{image_id}.txt"
    else:
        raise ValueError(f"Unsupported split: {split}")

    if SAVE_AS_JPG:
        cv2.imwrite(
            str(img_out_path),
            img_resized,
            [cv2.IMWRITE_JPEG_QUALITY, JPG_QUALITY]
        )
    else:
        cv2.imwrite(
            str(img_out_path),
            img_resized,
            [cv2.IMWRITE_PNG_COMPRESSION, 0]
        )

    yolo_lines = build_yolo_lines(
        boxes=boxes,
        orig_w=orig_w,
        orig_h=orig_h,
        img_size=IMG_SIZE,
    )

    with open(lbl_out_path, "w") as f:
        f.write("\n".join(yolo_lines))


def process_one_dicom(record):
    t0 = time.perf_counter()

    sample_id = str(record["sample_id"])
    dicom_path = Path(record["path"])
    split = split_map.get(sample_id)
    if split is None:
        return {
            "image_id": sample_id,
            "ok": False,
            "reason": "not_in_split",
            "elapsed": 0.0,
            "num_saved": 0,
        }

    boxes = ann_map.get(sample_id, [])

    # đọc ảnh gốc
    img_raw = read_dicom_to_uint8(dicom_path)

    try:
        # 1) luôn lưu ảnh gốc
        save_image_and_label(
            img=img_raw,
            boxes=boxes,
            image_id=sample_id,
            split=split,
        )

        num_saved = 1

        # 2) nếu là train thì lưu thêm ảnh augment
        if split == "train" and len(boxes) > 0:
            img_aug, boxes_aug = apply_dicom_augmentation(
                img_raw.copy(),
                boxes,
                split,
            )

            save_image_and_label(
                img=img_aug,
                boxes=boxes_aug,
                image_id=f"{sample_id}_aug",
                split=split,
            )
            num_saved += 1

        elapsed = time.perf_counter() - t0
        return {
            "image_id": sample_id,
            "ok": True,
            "reason": "done",
            "elapsed": elapsed,
            "num_saved": num_saved,
        }

    except Exception as e:
        elapsed = time.perf_counter() - t0
        return {
            "image_id": sample_id,
            "ok": False,
            "reason": str(e),
            "elapsed": elapsed,
            "num_saved": 0,
        }

# Chạy export song song + ETA

In [ ]:
max_workers = min(8, os.cpu_count() or 4)
print("max_workers =", max_workers)

total_files = len(dicom_records_selected)
start_all = time.perf_counter()

done = 0
ok_count = 0
times = []

log_every = 20

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_one_dicom, record) for record in dicom_records_selected]

    for future in as_completed(futures):
        result = future.result()
        done += 1

        if result["ok"]:
            ok_count += 1
            times.append(result["elapsed"])

        if done % log_every == 0 or done == total_files:
            elapsed_all = time.perf_counter() - start_all
            avg_per_file = elapsed_all / done
            speed = done / elapsed_all if elapsed_all > 0 else 0.0
            remaining = total_files - done
            eta_sec = remaining * avg_per_file

            avg_worker = np.mean(times) if len(times) > 0 else 0.0
            p50_worker = np.median(times) if len(times) > 0 else 0.0

            print(
                f"[{done}/{total_files}] "
                f"ok={ok_count} | "
                f"wall={elapsed_all:.1f}s | "
                f"speed={speed:.2f} img/s | "
                f"avg_wall/file={avg_per_file:.3f}s | "
                f"avg_worker={avg_worker:.3f}s | "
                f"p50_worker={p50_worker:.3f}s | "
                f"ETA={eta_sec:.1f}s"
            )

total_elapsed = time.perf_counter() - start_all
print("\nDONE")
print(f"Processed: {done}/{total_files}")
print(f"Success:   {ok_count}")
print(f"Total time: {total_elapsed:.2f}s")
print(f"Overall speed: {done / total_elapsed:.2f} img/s")

In [ ]:
img_ext = "*.jpg" if SAVE_AS_JPG else "*.png"

print("Used source train images:", len(train_ids))
print("Used source val images:", len(val_ids))
print("Used source test images:", len(test_ids))
print("Train images:", len(list(IMG_TRAIN_DIR.glob(img_ext))))
print("Val images:", len(list(IMG_VAL_DIR.glob(img_ext))))
print("Test images:", len(list(IMG_TEST_DIR.glob(img_ext))))
print("Train labels:", len(list(LBL_TRAIN_DIR.glob("*.txt"))))
print("Val labels:", len(list(LBL_VAL_DIR.glob("*.txt"))))
print("Test labels:", len(list(LBL_TEST_DIR.glob("*.txt"))))

# Tạo data.yaml

In [ ]:
data_yaml = {
    "path": str(OUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)}
}

yaml_path = OUT_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Saved:", yaml_path)
print(yaml_path.read_text())

In [ ]:
import matplotlib.pyplot as plt
for i in range(10):
    img_ext = ".jpg" if SAVE_AS_JPG else ".png"
    sample_image_path = list(IMG_TRAIN_DIR.glob(f"*{img_ext}"))[i]
    sample_label_path = LBL_TRAIN_DIR / f"{sample_image_path.stem}.txt"
    
    img = cv2.imread(str(sample_image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    h, w = img.shape[:2]
    
    if sample_label_path.exists():
        lines = sample_label_path.read_text().strip().splitlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
    
            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:])
    
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
    
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(
                img,
                ID2CLASS[cls_id],
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 0),
                2
            )
    
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.title(sample_image_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device)

# model nhỏ để test nhanh
model = YOLO("yolov8n.pt")

model.train(
    data=str(yaml_path),
    epochs=50,               # test nhanh, có thể đổi 1 / 3 / 10
    imgsz=IMG_SIZE,
    batch=8,                # nếu thiếu VRAM thì giảm xuống 4
    device=device,
    project=str(OUT_ROOT / "runs"),
    name="yolov8n_multiclass_test",
    exist_ok=True
)

In [ ]:
# Eval trên test split đã được tạo từ bước gộp train + test rồi stratified split.
# Không preprocess lại test CSV ở đây nữa, tránh ghi đè test split mới.

import torch
from ultralytics import YOLO

EVAL_BATCH = 8
TRAIN_RUN_NAME = "yolov8n_multiclass_test"  # sửa nếu run name khác
BEST_WEIGHT_PATH = OUT_ROOT / "runs" / TRAIN_RUN_NAME / "weights" / "best.pt"

if not BEST_WEIGHT_PATH.exists():
    raise FileNotFoundError(f"Không thấy best weight: {BEST_WEIGHT_PATH}")

best_model = YOLO(str(BEST_WEIGHT_PATH))
device = 0 if torch.cuda.is_available() else "cpu"
metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=IMG_SIZE,
    batch=EVAL_BATCH,
    device=device,
    verbose=True,
)

print("Test metrics:")
print({
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
})